In [4]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[1] # go up n levels (adjust as needed)
sys.path.append(str(ROOT))

from config import PROJECT_ROOT, APT_ROOT
from apt_project import *

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import GradientBoostingRegressor
import numpy as np

In [7]:
# Define feature columns
feature_cols = ['year', 'month', 'industry_code', 'event_type', 'event_subtype', 'motive', 'actor_type']

# Identify Categorical vs Numeric
categorical_cols = ['industry_code', 'event_type', 'event_subtype', 'motive', 'actor_type']
numeric_cols =['year', 'month']

In [8]:
# Preprocess with one-hot encoding (fit on all data)
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', 'passthrough', numeric_cols)
    ]
)

# Fit preprocessor on full dataset
preprocessor.fit(events_df[feature_cols])

,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,None
,sparse_output,True


In [9]:
# Filter out NaN values for the respective training sets
train_df_origin = events_df[events_df['origin_risk_norm'].notna()]
train_df_victim = events_df[events_df['victim_risk_norm'].notna()]

In [10]:
# Target Variables (y)
y_origin = train_df_origin['origin_risk_norm']
y_victim = train_df_victim['victim_risk_norm']

# Input Features (X)
X_origin_raw = train_df_origin[feature_cols]
X_origin = preprocessor.transform(X_origin_raw)
X_victim_raw = train_df_victim[feature_cols]
X_victim = preprocessor.transform(X_victim_raw)

In [11]:
# Train models
origin_model = GradientBoostingRegressor()
victim_model = GradientBoostingRegressor()

origin_model.fit(X_origin, y_origin)
victim_model.fit(X_victim, y_victim)

,loss,'squared_error'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [12]:
# Predict for all events
X_all = preprocessor.transform(events_df[feature_cols])

events_df['ml_origin_risk'] = origin_model.predict(X_all)
events_df['ml_target_risk'] = victim_model.predict(X_all)

In [14]:
#events_df